# 配套实践 11-01：切分 Action Chunk 并滚动执行

本练习从一条连续示范轨迹切出 observation horizon 与 prediction horizon，随后用带轻微偏差的重叠动作块观察 temporal ensemble，最后比较整块开环执行和 receding horizon 在突发扰动后的差别。依赖：NumPy、Matplotlib；CPU 即可运行。

<a href="https://qi-robotics.github.io/robot-world-model-tutorial/intermediate/11-demonstrations-to-action-sequences/" target="_blank">在新标签页返回课程正文</a>

In [ ]:
import numpy as np  # 构造示范轨迹、动作窗口与滚动执行
import matplotlib.pyplot as plt  # 绘制切窗、动作融合和闭环结果
np.random.seed(111)  # 固定动作块中的模拟预测误差
plt.rcParams["figure.dpi"] = 120  # 提高笔记本图像显示清晰度

## 1. 从完整轨迹建立因果训练样本

示范是一段平滑关节位置，动作定义为相邻位置之差。锚点时刻左侧 8 步作为观察历史，从锚点开始的 12 步动作作为标签。这里先展示一个窗口，再批量切出全部有效样本。

In [ ]:
time_steps = np.arange(80)  # 建立八十个离散轨迹时刻
joint_positions = 0.6 * np.sin(time_steps / 11.0) + 0.012 * time_steps  # 构造同时包含周期运动和缓慢趋势的关节位置
actions = np.diff(joint_positions)  # 把动作定义为相邻时刻的关节位置增量
observation_horizon = 8  # 设置 Context 能读取的历史长度
prediction_horizon = 12  # 设置 Action Model 一次预测的动作块长度
anchor_time = 35  # 选择一个中间时刻展示切窗关系
history_indices = np.arange(anchor_time - observation_horizon + 1, anchor_time + 1)  # 找到锚点左侧含当前时刻的观察索引
action_indices = np.arange(anchor_time, anchor_time + prediction_horizon)  # 找到从锚点开始的未来动作标签索引
history_windows = []  # 准备保存所有完整观察历史
action_chunks = []  # 准备保存所有完整未来动作块
anchor_indices = []  # 准备保存每个训练样本的锚点
for current_time in range(observation_horizon - 1, len(actions) - prediction_horizon + 1):  # 只遍历历史和未来均完整的锚点
    history_windows.append(joint_positions[current_time - observation_horizon + 1:current_time + 1])  # 切出当前样本的观察历史
    action_chunks.append(actions[current_time:current_time + prediction_horizon])  # 切出当前样本的未来动作块
    anchor_indices.append(current_time)  # 保存该样本在原始轨迹中的锚点
history_windows = np.stack(history_windows)  # 把观察窗口组成样本乘历史长度的数组
action_chunks = np.stack(action_chunks)  # 把动作块组成样本乘预测长度的数组
fig, axis = plt.subplots(figsize=(10, 4.0))  # 创建原始轨迹及单个训练窗口图
axis.plot(time_steps, joint_positions, color="#94a3b8", label="Full demonstration")  # 绘制完整示范关节位置
axis.plot(history_indices, joint_positions[history_indices], color="#2563eb", linewidth=4, label="Observation horizon")  # 高亮 Context 使用的历史
future_positions = joint_positions[anchor_time] + np.cumsum(actions[action_indices])  # 把未来增量动作还原为位置便于展示
axis.plot(np.arange(anchor_time + 1, anchor_time + prediction_horizon + 1), future_positions, color="#16a34a", linewidth=4, label="Action chunk label")  # 高亮未来动作标签覆盖的位置
axis.axvline(anchor_time, color="#ea580c", linestyle="--", label="Anchor time")  # 标出历史与未来的因果边界
axis.set(title=f"One trajectory creates {len(action_chunks)} training windows", xlabel="Time step", ylabel="Joint position")  # 标注轨迹、时间和样本数量
axis.legend()  # 显示历史、动作块与锚点图例
axis.grid(alpha=0.2)  # 添加淡网格帮助读取时间范围
fig.tight_layout()  # 调整图像边距
plt.show()  # 显示从连续示范切出训练样本的过程

**怎样理解结果：** 蓝线全部位于锚点及其左侧，是模型产生 Context 时允许读取的历史；绿线位于锚点之后，是监督动作块。一个锚点产生一个样本，相邻锚点的窗口会大量重叠。因此必须先按整条轨迹划分训练集和验证集，不能把这些高度相似的窗口随机拆开。

## 2. 融合指向同一时刻的重叠预测

我们模拟模型在每个锚点都预测一个 12 步动作块。不同块带有轻微整体偏差和随预测距离增加的噪声。最新块策略只取最近锚点给出的动作；temporal ensemble 则对指向同一执行时刻的多个预测按新旧程度加权。

In [ ]:
chunk_length = 12  # 设置每次模型输出十二步动作
prediction_starts = np.arange(10, 68)  # 设置可以产生完整动作块的决策锚点
predicted_chunks = {}  # 准备按锚点保存模拟模型输出
for start_time in prediction_starts:  # 逐个锚点产生带误差的动作块
    true_chunk = actions[start_time:start_time + chunk_length]  # 取出该锚点对应的真实示范动作
    chunk_bias = np.random.normal(scale=0.008)  # 为整个动作块加入一次共同偏差
    horizon_noise = np.random.normal(scale=np.linspace(0.002, 0.015, chunk_length))  # 让远期预测具有更大噪声
    predicted_chunks[start_time] = true_chunk + chunk_bias + horizon_noise  # 保存当前锚点的模拟预测结果
execution_times = np.arange(21, 68)  # 选择同时被多个动作块覆盖的执行时间范围
latest_actions = []  # 保存只使用最新动作块时的当前命令
ensemble_actions = []  # 保存融合重叠动作块后的当前命令
contributor_counts = []  # 保存每个执行时刻参与融合的块数量
decay_rate = 0.35  # 设置旧预测随年龄衰减的速度
for execution_time in execution_times:  # 逐个执行时刻汇总所有可用预测
    contributors = []  # 暂存指向当前时刻的预测值与锚点
    for start_time, chunk in predicted_chunks.items():  # 遍历此前产生的全部动作块
        offset = execution_time - start_time  # 计算当前时刻在该动作块中的相对位置
        if 0 <= offset < chunk_length and start_time <= execution_time:  # 只保留确实覆盖当前时刻的因果预测
            contributors.append((start_time, chunk[offset]))  # 保存预测来源时刻和对应动作值
    newest_start = max(start_time for start_time, _ in contributors)  # 找到信息最新的预测块
    latest_value = [value for start_time, value in contributors if start_time == newest_start][0]  # 读取最新块给出的当前动作
    ages = np.array([execution_time - start_time for start_time, _ in contributors])  # 计算各预测距离当前执行时刻的年龄
    weights = np.exp(-decay_rate * ages)  # 为更旧预测分配更小的指数权重
    values = np.array([value for _, value in contributors])  # 整理所有候选动作值
    latest_actions.append(latest_value)  # 保存最新块策略的动作
    ensemble_actions.append(float(np.sum(weights * values) / np.sum(weights)))  # 保存归一化加权后的动作
    contributor_counts.append(len(contributors))  # 记录当前时刻参与融合的块数
latest_actions = np.array(latest_actions)  # 把最新块动作转换为 NumPy 数组
ensemble_actions = np.array(ensemble_actions)  # 把融合动作转换为 NumPy 数组
true_actions = actions[execution_times]  # 取出同一时间范围的真实示范动作
fig, axes = plt.subplots(2, 1, figsize=(10, 5.7), sharex=True)  # 创建动作曲线与贡献数量两个坐标轴
axes[0].plot(execution_times, true_actions, color="#172033", linewidth=2.5, label="Demonstration")  # 绘制无模拟误差的目标动作
axes[0].plot(execution_times, latest_actions, color="#ea580c", alpha=0.8, label="Newest chunk only")  # 绘制只使用最新块的抖动命令
axes[0].plot(execution_times, ensemble_actions, color="#2563eb", linewidth=2, label="Temporal ensemble")  # 绘制融合后的平滑命令
axes[0].set(ylabel="Action increment", title="Overlapping chunks provide several predictions for one time step")  # 标注动作语义与图像主题
axes[0].legend()  # 显示三条动作曲线图例
axes[1].bar(execution_times, contributor_counts, color="#94a3b8")  # 显示每个时刻参与融合的预测块数量
axes[1].set(xlabel="Execution time", ylabel="Chunk count")  # 标注执行时间和重叠块数量
fig.tight_layout()  # 调整两幅子图间距
plt.show()  # 显示重叠动作块与 temporal ensemble 的效果

**怎样理解结果：** 橙线每一步都来自新动作块的第一个位置，容易继承块间偏差而抖动；蓝线融合了多个锚点对同一时刻的预测，通常更接近连续趋势。下图说明稳定区间可同时获得约 12 个预测。融合越强不一定越好：旧 Context 的权重过大会产生滞后，突发变化时应更依赖新观测。

## 3. 扰动后为什么需要 receding horizon

最后使用一维移动任务比较两种执行方式。开环方案在开始时预测整段动作并全部执行；滚动方案每 5 步重新读取当前位置，重新计算剩余动作。第 12 步加入一次向后的外部扰动。

In [ ]:
target_position = 1.0  # 设置机器人需要到达的一维目标位置
total_steps = 30  # 设置完整控制过程的离散步数
replan_interval = 5  # 设置滚动方案每五步重新规划一次
disturbance_step = 12  # 设置外部扰动发生的时刻
disturbance_value = -0.25  # 设置扰动把机器人向后推的距离
open_loop_position = 0.0  # 初始化开环执行位置
receding_position = 0.0  # 初始化滚动重规划位置
open_loop_action = (target_position - open_loop_position) / total_steps  # 在开始时计算整段固定开环动作
receding_action = 0.0  # 初始化滚动方案的当前动作
open_loop_trace = [open_loop_position]  # 保存开环执行的完整位置轨迹
receding_trace = [receding_position]  # 保存滚动方案的完整位置轨迹
for step_index in range(total_steps):  # 按时间推进两个执行方案
    if step_index % replan_interval == 0:  # 到达滚动方案的重新规划时刻
        remaining_steps = total_steps - step_index  # 计算当前还剩多少控制步
        receding_action = (target_position - receding_position) / remaining_steps  # 根据新位置重新分配剩余位移
    open_loop_position += open_loop_action  # 开环方案始终执行开始时计算的固定动作
    receding_position += receding_action  # 滚动方案执行最近一次规划得到的动作
    if step_index == disturbance_step:  # 检查当前时刻是否发生外部扰动
        open_loop_position += disturbance_value  # 给开环执行位置加入相同向后扰动
        receding_position += disturbance_value  # 给滚动执行位置加入相同向后扰动
    open_loop_trace.append(open_loop_position)  # 记录当前开环位置
    receding_trace.append(receding_position)  # 记录当前滚动位置
fig, axis = plt.subplots(figsize=(9.5, 4.0))  # 创建两种执行方案的位置曲线
axis.plot(open_loop_trace, color="#ea580c", linewidth=2.5, label="Open-loop full chunk")  # 绘制扰动后不修正的开环轨迹
axis.plot(receding_trace, color="#2563eb", linewidth=2.5, label="Receding horizon")  # 绘制周期性重新规划的轨迹
axis.axhline(target_position, color="#16a34a", linestyle="--", label="Target")  # 标出期望最终位置
axis.axvline(disturbance_step + 1, color="#dc2626", linestyle=":", label="Disturbance")  # 标出扰动进入状态后的时刻
for replan_step in range(0, total_steps + 1, replan_interval):  # 遍历滚动方案的规划时刻
    axis.axvline(replan_step, color="#94a3b8", alpha=0.25)  # 用淡线显示每次重新读取状态的时刻
axis.set(title="Replanning corrects deviations after new observations", xlabel="Control step", ylabel="Position")  # 标注闭环纠错图像含义
axis.legend()  # 显示开环、滚动、目标和扰动图例
axis.grid(alpha=0.2)  # 添加淡网格帮助读取最终误差
fig.tight_layout()  # 调整图像边距
plt.show()  # 显示扰动后的两种执行结果

**怎样理解结果：** 两种方案在扰动前沿相同路径前进。开环方案没有读取受扰后的新位置，后续仍执行旧动作，因此最终保留约 0.25 的误差；滚动方案在下一次规划时看到偏差，把剩余距离重新分配到后续动作，最终回到目标附近。

**本练习的结论：** prediction horizon 决定模型能表达多长的连贯动作，execution horizon 决定多久重新观察。Action Chunk 与闭环并不矛盾：可以预测较长动作块，只执行较短前缀，再滚动生成。